In [0]:
import math

CATALOG   = "e-commerce"
SILVER_DB = "silver"
GOLD_DB   = "gold"

In [0]:
spark.sql(f"USE CATALOG `{CATALOG}`")
spark.sql(f"CREATE DATABASE IF NOT EXISTS {GOLD_DB}")

spark.table(f"{SILVER_DB}.fact_sales").createOrReplaceTempView("fact_sales")
spark.table(f"{SILVER_DB}.dim_stores").createOrReplaceTempView("dim_stores")
spark.table(f"{SILVER_DB}.dim_calendar").createOrReplaceTempView("dim_calendar")
print("Gold DB ready | Silver views registered")

In [0]:
df_store_revenue = spark.sql("""
    SELECT
        s.store_sk,
        s.store_id,
        s.store_name,
        s.city,
        s.country,
        s.store_type,
        s.channel,

        COUNT(f.order_id)                               AS total_orders,
        SUM(f.quantity)                                 AS total_units_sold,
        ROUND(SUM(f.gross_sales),   2)                  AS gross_sales,
        ROUND(SUM(f.discount_amount), 2)                AS total_discount,
        ROUND(SUM(f.revenue),       2)                  AS total_revenue,
        ROUND(SUM(f.cost),          2)                  AS total_cost,
        ROUND(SUM(f.profit),        2)                  AS total_profit,
        ROUND(AVG(f.profit_margin)*100, 2)              AS avg_profit_margin_pct,
        ROUND(SUM(f.revenue) / COUNT(f.order_id), 2)   AS aov,

        ROUND(
            SUM(f.revenue) * 100.0
            / SUM(SUM(f.revenue)) OVER (), 2
        )                                               AS revenue_share_pct,

        RANK() OVER (ORDER BY SUM(f.revenue) DESC)      AS revenue_rank,
        RANK() OVER (ORDER BY SUM(f.profit)  DESC)      AS profit_rank

    FROM fact_sales f
    JOIN dim_stores s ON f.store_sk = s.store_sk
    WHERE f.is_valid = 1
    GROUP BY
        s.store_sk, s.store_id, s.store_name,
        s.city, s.country, s.store_type, s.channel
    ORDER BY total_revenue DESC
""")

print("TOP 20 STORES BY REVENUE")
df_store_revenue.show(20, truncate=False)

row_count = df_store_revenue.count()
num_partitions = max(2, math.ceil(row_count / 10000))

df_store_revenue.repartition(num_partitions).write \
    .format("delta").mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{GOLD_DB}.store_revenue_ranking")

print(f"gold.store_revenue_ranking written - {row_count} stores ({num_partitions} partitions)")

In [0]:
df_geo = spark.sql("""
    SELECT
        s.country,
        s.city,

        COUNT(DISTINCT s.store_sk)                      AS store_count,
        COUNT(f.order_id)                               AS total_orders,
        SUM(f.quantity)                                 AS total_units,
        ROUND(SUM(f.revenue),  2)                       AS total_revenue,
        ROUND(SUM(f.profit),   2)                       AS total_profit,
        ROUND(AVG(f.profit_margin)*100, 2)              AS avg_margin_pct,
        ROUND(SUM(f.revenue) / COUNT(DISTINCT s.store_sk), 2) AS revenue_per_store,
        ROUND(SUM(f.revenue) / COUNT(f.order_id), 2)   AS aov,

        ROUND(
            SUM(f.revenue) * 100.0
            / SUM(SUM(f.revenue)) OVER (PARTITION BY s.country), 2
        )                                               AS city_share_in_country_pct,

        RANK() OVER (ORDER BY SUM(f.revenue) DESC)      AS overall_revenue_rank,
        RANK() OVER (PARTITION BY s.country ORDER BY SUM(f.revenue) DESC) AS rank_in_country

    FROM fact_sales f
    JOIN dim_stores s ON f.store_sk = s.store_sk
    WHERE f.is_valid = 1
    GROUP BY s.country, s.city
    ORDER BY total_revenue DESC
""")

print("CITY / COUNTRY PERFORMANCE")
df_geo.show(30, truncate=False)

print("COUNTRY SUMMARY")
spark.sql("""
    SELECT
        s.country,
        COUNT(DISTINCT s.store_sk)      AS stores,
        ROUND(SUM(f.revenue), 2)        AS total_revenue,
        ROUND(SUM(f.profit), 2)         AS total_profit,
        ROUND(AVG(f.profit_margin)*100,2) AS avg_margin_pct,
        ROUND(SUM(f.revenue)*100.0 / SUM(SUM(f.revenue)) OVER (), 2) AS revenue_share_pct
    FROM fact_sales f
    JOIN dim_stores s ON f.store_sk = s.store_sk
    WHERE f.is_valid = 1
    GROUP BY s.country
    ORDER BY total_revenue DESC
""").show(truncate=False)

row_count = df_geo.count()
num_partitions = max(2, math.ceil(row_count / 10000))

df_geo.repartition(num_partitions).write \
    .format("delta").mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{GOLD_DB}.city_country_performance")

print(f"gold.city_country_performance written - {row_count} rows")

In [0]:
df_store_type = spark.sql("""
    SELECT
        s.store_type,
        s.channel,

        COUNT(DISTINCT s.store_sk)                      AS store_count,
        COUNT(f.order_id)                               AS total_orders,
        SUM(f.quantity)                                 AS total_units,
        ROUND(SUM(f.revenue),    2)                     AS total_revenue,
        ROUND(SUM(f.profit),     2)                     AS total_profit,
        ROUND(AVG(f.profit_margin)*100, 2)              AS avg_margin_pct,
        ROUND(SUM(f.revenue) / COUNT(DISTINCT s.store_sk), 2) AS revenue_per_store,
        ROUND(COUNT(f.order_id) / COUNT(DISTINCT s.store_sk), 2) AS orders_per_store,
        ROUND(SUM(f.revenue) / COUNT(f.order_id), 2)   AS aov,

        ROUND(
            SUM(f.revenue) * 100.0
            / SUM(SUM(f.revenue)) OVER (), 2
        )                                               AS revenue_share_pct,

        RANK() OVER (ORDER BY SUM(f.revenue) DESC)      AS revenue_rank

    FROM fact_sales f
    JOIN dim_stores s ON f.store_sk = s.store_sk
    WHERE f.is_valid = 1
    GROUP BY s.store_type, s.channel
    ORDER BY total_revenue DESC
""")

print("STORE TYPE PERFORMANCE")
df_store_type.show(truncate=False)

row_count = df_store_type.count()
num_partitions = max(2, math.ceil(row_count / 10000))

df_store_type.repartition(num_partitions).write \
    .format("delta").mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{GOLD_DB}.store_type_performance")

print(f"gold.store_type_performance written - {row_count} rows")

In [0]:
df_underperforming = spark.sql("""
    WITH store_metrics AS (
        SELECT
            s.store_sk,
            s.store_id,
            s.store_name,
            s.city,
            s.country,
            s.store_type,
            s.channel,

            COUNT(f.order_id)                               AS total_orders,
            ROUND(SUM(f.revenue),  2)                       AS total_revenue,
            ROUND(SUM(f.profit),   2)                       AS total_profit,
            ROUND(AVG(f.profit_margin)*100, 2)              AS avg_margin_pct,

            AVG(SUM(f.revenue)) OVER (PARTITION BY s.store_type) AS type_avg_revenue,
            AVG(AVG(f.profit_margin)) OVER (PARTITION BY s.store_type) AS type_avg_margin,

            PERCENT_RANK() OVER (ORDER BY SUM(f.revenue) ASC) AS revenue_pct_rank

        FROM fact_sales f
        JOIN dim_stores s ON f.store_sk = s.store_sk
        WHERE f.is_valid = 1
        GROUP BY
            s.store_sk, s.store_id, s.store_name,
            s.city, s.country, s.store_type, s.channel
    )
    SELECT
        store_id,
        store_name,
        city,
        country,
        store_type,
        channel,
        total_orders,
        total_revenue,
        total_profit,
        avg_margin_pct,
        ROUND(type_avg_revenue, 2)      AS store_type_avg_revenue,
        ROUND(type_avg_margin * 100, 2) AS store_type_avg_margin_pct,

        ROUND(total_revenue - type_avg_revenue, 2) AS gap_vs_type_avg,
        ROUND(total_revenue / type_avg_revenue * 100, 2) AS pct_of_type_avg,

        CASE
            WHEN total_revenue < type_avg_revenue * 0.50 THEN 'Critical Underperformer'
            WHEN total_revenue < type_avg_revenue * 0.75 THEN 'Underperformer'
            WHEN total_revenue < type_avg_revenue * 0.90 THEN 'Below Average'
            WHEN total_revenue > type_avg_revenue * 1.25 THEN 'Top Performer'
            ELSE 'Average'
        END AS performance_label

    FROM store_metrics
    ORDER BY total_revenue ASC
""")

print("UNDERPERFORMING STORES")
df_underperforming.filter(
    df_underperforming.performance_label.isin(
        ["Critical Underperformer", "Underperformer", "Below Average"]
    )
).show(30, truncate=False)

print("PERFORMANCE DISTRIBUTION")
df_underperforming.groupBy("performance_label").count().orderBy("count", ascending=False).show()

row_count = df_underperforming.count()
num_partitions = max(2, math.ceil(row_count / 10000))

df_underperforming.repartition(num_partitions).write \
    .format("delta").mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{GOLD_DB}.underperforming_stores")

print(f"gold.underperforming_stores written - {row_count} stores ({num_partitions} partitions)")

In [0]:
df_channel = spark.sql("""
    SELECT
        s.channel,
        c.year,
        c.year_quarter,
        c.year_month,
        c.month_name,

        COUNT(DISTINCT s.store_sk)                      AS stores_active,
        COUNT(f.order_id)                               AS total_orders,
        SUM(f.quantity)                                 AS total_units,
        ROUND(SUM(f.revenue),    2)                     AS total_revenue,
        ROUND(SUM(f.profit),     2)                     AS total_profit,
        ROUND(AVG(f.profit_margin)*100, 2)              AS avg_margin_pct,
        ROUND(SUM(f.revenue) / COUNT(f.order_id), 2)   AS aov,

        ROUND(
            SUM(f.revenue) * 100.0
            / SUM(SUM(f.revenue)) OVER (PARTITION BY c.year_month), 2
        )                                               AS channel_share_pct

    FROM fact_sales   f
    JOIN dim_stores   s ON f.store_sk = s.store_sk
    JOIN dim_calendar c ON f.date_sk  = c.date_sk
    WHERE f.is_valid = 1
    GROUP BY s.channel, c.year, c.year_quarter, c.year_month, c.month_name
    ORDER BY c.year_month, s.channel
""")

print("CHANNEL COMPARISON (monthly)")
df_channel.show(30, truncate=False)

print("CHANNEL OVERALL SUMMARY")
spark.sql("""
    SELECT
        s.channel,
        COUNT(DISTINCT s.store_sk)  AS stores,
        COUNT(f.order_id)           AS total_orders,
        ROUND(SUM(f.revenue),  2)   AS total_revenue,
        ROUND(SUM(f.profit),   2)   AS total_profit,
        ROUND(AVG(f.profit_margin)*100, 2) AS avg_margin_pct,
        ROUND(SUM(f.revenue)*100.0 / SUM(SUM(f.revenue)) OVER(), 2) AS revenue_share_pct
    FROM fact_sales f
    JOIN dim_stores s ON f.store_sk = s.store_sk
    WHERE f.is_valid = 1
    GROUP BY s.channel
    ORDER BY total_revenue DESC
""").show(truncate=False)

row_count = df_channel.count()
num_partitions = max(2, math.ceil(row_count / 10000))

df_channel.repartition(num_partitions).write \
    .format("delta").mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{GOLD_DB}.channel_comparison")

print(f"gold.channel_comparison written - {row_count} rows")

In [0]:
print("GOLD - Store Performance COMPLETE")
print("  gold.store_revenue_ranking    - Store revenue/profit")
print("  gold.city_country_performance - City/country best")
print("  gold.store_type_performance   - Mall/Retail/Online")
print("  gold.underperforming_stores   - Underperformers")
print("  gold.channel_comparison       - Physical vs Digital")
print()

for tbl in ["store_revenue_ranking", "city_country_performance", "store_type_performance", "underperforming_stores", "channel_comparison"]:
    cnt = spark.table(f"{GOLD_DB}.{tbl}").count()
    print(f"   gold.{tbl:<30} - {cnt:,} rows")